# 📦 Phase 1: Data Engineering
**วิชา:** Introduction to Natural Language Processing  
**Domain:** Restaurant Reviews (Yelp.com)  
**Task:** Sentiment Analysis (Positive / Negative)

---

## 🔧 Install & Import Libraries

### 🔍 อธิบาย: ติดตั้ง Libraries

Cell นี้ติดตั้ง libraries ทั้งหมดที่จำเป็นสำหรับโปรเจกต์ครั้งเดียว:
- **`google-search-results`** — apify เพื่อดึง Restaurant Data
- **`pandas`** — จัดการข้อมูลในรูปแบบตาราง (DataFrame)
- **`nltk`** — ชุดเครื่องมือ NLP สำหรับ tokenization
- **`spacy`** — NLP library หลักที่ใช้ lemmatization และ NER
- **`datasets`** — HuggingFace library สำหรับโหลด Yelp Polarity baseline dataset

In [1]:
# ติดตั้ง libraries ที่จำเป็น (รันครั้งเดียว)
!pip install pandas nltk spacy datasets
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.7 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


### 🔍 อธิบาย: Import และโหลด Libraries

Import libraries ที่ติดตั้งแล้วเข้ามาใช้งาน:
- **`GoogleSearch`** จาก Apify เพื่อ Scrape Data
- **`punkt` / `punkt_tab`** — model สำหรับตัด sentence และ word ของ NLTK
- **`stopwords`** — รายการคำที่ไม่มีความหมาย เช่น `the`, `is`, `a` เพื่อใช้กรองออก

ถ้ารันสำเร็จจะเห็น `✅ Libraries loaded successfully!`

In [2]:
import pandas as pd
import random
import re
import json
import nltk
import spacy
from datasets import load_dataset

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

print('✅ Libraries loaded successfully!')


/opt/anaconda3/envs/nlp_intro/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries loaded successfully!


[nltk_data] Downloading package punkt to /Users/visanu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/visanu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/visanu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---
## 📚 1.1A — Standard Dataset (Baseline)
ใช้ **Yelp Review Polarity** จาก HuggingFace เป็น Baseline  
- Label: `0` = Negative (1-2 ดาว), `1` = Positive (4-5 ดาว)
- ขนาด: ~560,000 reviews (เราจะสุ่มใช้แค่ 1,000)

### 🔍 อธิบาย: โหลด Standard Dataset (Baseline)

โหลด **Yelp Review Polarity** จาก HuggingFace ซึ่งเป็น dataset มาตรฐานที่ได้รับการ label แล้ว:
- Dataset นี้มี ~560,000 reviews แบ่งเป็น Positive (4-5 ดาว) และ Negative (1-2 ดาว)
- ใช้ `random.seed(42)` เพื่อให้ผลการสุ่มเหมือนกันทุกครั้งที่รัน (Reproducibility)
- สุ่มเลือกแค่ **1,000 rows** เพื่อความเร็วในการ train โมเดล
- บันทึกลงใน `df_baseline` พร้อม column: `text`, `label`, `source`

**label:** `0` = Negative, `1` = Positive

In [3]:
# โหลด Standard Dataset
dataset = load_dataset('yelp_polarity', split='train')

# สุ่ม 1,000 รายการเพื่อความเร็ว
import random
random.seed(42)
indices = random.sample(range(len(dataset)), 1000)
sample = dataset.select(indices)

# แปลงเป็น DataFrame
df_baseline = pd.DataFrame({
    'text': sample['text'],
    'label': sample['label'],   # 0 = Negative, 1 = Positive
    'source': 'huggingface_baseline'
})

print(f'✅ Baseline dataset loaded: {len(df_baseline)} rows')
print(f'Label distribution:\n{df_baseline["label"].value_counts()}')
df_baseline.head(3)

✅ Baseline dataset loaded: 1000 rows
Label distribution:
label
0    510
1    490
Name: count, dtype: int64


,text,label,source
0,All the steaks are pre-made .. Once you order ...,0,huggingface_baseline
1,I was craving Dairy Queen for quite awhile. W...,0,huggingface_baseline
2,Un endroit qui ressemble plus \u00e0 un bar sp...,0,huggingface_baseline


---
## 🌐 1.1B — Web Scraping (บังคับ): Google Maps Reviews

**แหล่งข้อมูล:** ไฟล์ CSV 2 ชุดที่ดึงด้วย [Apify Google Maps Reviews Scraper](https://apify.com/compass/google-maps-reviews-scraper)

| ไฟล์ | ร้าน | Reviews |
|------|------|---------|
| `dataset_Google-Maps-Reviews-Scraper_2026-03-10_06-16-58-529.csv` | Tokyo Hibachi Catering, Manhatta, Thai Villa, Boucherie Union Square, The Landing, Eddie's Pizza | 525 |
| `dataset_Google-Maps-Reviews-Scraper_2026-03-10_06-34-10-986.csv` | Eataly, ZAZA, The Smith, Le Bernardin | 400 |

**Labeling Strategy:**
- Star 1-2 → **Negative (label = 0)**
- Star 3   → ข้าม (Neutral — ตีความไม่ชัดเจน)
- Star 4-5 → **Positive (label = 1)**

> **AI Audit Note:** ข้อมูลทั้งสองชุดเก็บสำเร็จจริงผ่าน Apify — ไม่ต้องใช้ mock data


### 🔍 อธิบาย: โหลดและรวม Google Maps Reviews จาก CSV 2 ไฟล์

- โหลด CSV ทั้ง 2 ชุดแล้ว `pd.concat()` รวมกัน → **925 rows** รวม
- กรอง `text` ที่เป็น null ออก (user ให้ดาวแต่ไม่เขียนอะไร) → ลด **217 rows**
- กรอง `stars == 3` ออก (Neutral) → ลด **31 rows**
- แปลง stars → label: 1-2 = 0 (Negative), 4-5 = 1 (Positive)
- เพิ่ม column `source = 'google_maps'`
- ผลลัพธ์สุดท้าย: **677 reviews** จาก 10 ร้านอาหาร


In [5]:
# ============================================================
# โหลด Google Maps Reviews จาก CSV 2 ไฟล์
# ============================================================

CSV_1 = 'dataset_Google-Maps-Reviews-Scraper_2026-03-10_06-16-58-529.csv'
CSV_2 = 'dataset_Google-Maps-Reviews-Scraper_2026-03-10_06-34-10-986.csv'

df_gmaps1 = pd.read_csv(CSV_1)
df_gmaps2 = pd.read_csv(CSV_2)
df_gmaps_raw = pd.concat([df_gmaps1, df_gmaps2], ignore_index=True)

print(f'📂 CSV 1: {len(df_gmaps1)} rows ({df_gmaps1["title"].nunique()} ร้าน)')
print(f'📂 CSV 2: {len(df_gmaps2)} rows ({df_gmaps2["title"].nunique()} ร้าน)')
print(f'📦 รวม:   {len(df_gmaps_raw)} rows')
print(f'\nร้านอาหารทั้งหมด ({df_gmaps_raw["title"].nunique()} ร้าน):')
print(df_gmaps_raw['title'].value_counts().to_string())
print(f'\nStar distribution (combined):')
print(df_gmaps_raw['stars'].value_counts().sort_index().to_string())


📂 CSV 1: 525 rows (6 ร้าน)
📂 CSV 2: 400 rows (4 ร้าน)
📦 รวม:   925 rows

ร้านอาหารทั้งหมด (10 ร้าน):
title
Tokyo Hibachi Catering    100
Manhatta                  100
Thai Villa                100
Boucherie Union Square    100
Eataly                    100
ZAZA                      100
The Smith                 100
Le Bernardin              100
The Landing                73
Eddie's Pizza              52

Star distribution (combined):
stars
1     44
2     16
3     39
4     71
5    755


### 🔍 อธิบาย: กรองและ Label ข้อมูล

| Star | Label | ความหมาย |
|------|-------|----------|
| 1-2 | 0 | Negative |
| 3   | — | ข้าม (Neutral) |
| 4-5 | 1 | Positive |

**เหตุผลที่ข้าม Star 3:** รีวิว 3 ดาวมีความหมายกลางๆ — ใส่ label Positive หรือ Negative ก็ผิดทั้งคู่  
การข้ามออกทำให้ classifier เรียนรู้จาก signal ที่ชัดเจนกว่า

> **Note:** rows ที่ `text` เป็น null ถูกกรองออกด้วย เพราะไม่มีเนื้อหาให้วิเคราะห์


In [6]:
# ============================================================
# กรอง + Label
# ============================================================

# 1. ลบ rows ที่ text ว่าง (user ให้ดาวแต่ไม่เขียนอะไร)
n_before = len(df_gmaps_raw)
df_gmaps = df_gmaps_raw.dropna(subset=['text']).copy()
print(f'ลบ null text: -{n_before - len(df_gmaps)} rows → เหลือ {len(df_gmaps)}')

# 2. ข้าม star 3 (Neutral)
n_before = len(df_gmaps)
df_gmaps = df_gmaps[df_gmaps['stars'] != 3].copy()
print(f'ลบ star 3:   -{n_before - len(df_gmaps)} rows → เหลือ {len(df_gmaps)}')

# 3. แปลง stars → label
df_gmaps['label'] = df_gmaps['stars'].apply(lambda s: 1 if s >= 4 else 0)

# 4. เพิ่ม source column
df_gmaps['source'] = 'google_maps'

# 5. จัดรูป DataFrame
df_scraped = df_gmaps[['text', 'label', 'source', 'title', 'stars']].copy()
df_scraped = df_scraped.rename(columns={'title': 'restaurant'})
df_scraped = df_scraped.reset_index(drop=True)

print(f'\n✅ Google Maps reviews พร้อมใช้งาน: {len(df_scraped)} rows')
print(f'\nLabel distribution:')
print(df_scraped['label'].value_counts())
print(f'\nต่อร้าน (Neg / Pos):')
print(df_scraped.groupby('restaurant')['label'].value_counts().unstack(fill_value=0))
df_scraped.head(3)


ลบ null text: -217 rows → เหลือ 708
ลบ star 3:   -31 rows → เหลือ 677

✅ Google Maps reviews พร้อมใช้งาน: 677 rows

Label distribution:
label
1    622
0     55
Name: count, dtype: int64

ต่อร้าน (Neg / Pos):
label                    0   1
restaurant                    
Boucherie Union Square   3  81
Eataly                  27  53
Eddie's Pizza            7  31
Le Bernardin             2  43
Manhatta                 5  58
Thai Villa               1  78
The Landing              4  53
The Smith                1  70
Tokyo Hibachi Catering   0  83
ZAZA                     5  72


,text,label,source,restaurant,stars
0,Kevin was amazing and the food was delicious.,1,google_maps,Tokyo Hibachi Catering,5
1,"Kevin was great! He was on time, entertaining ...",1,google_maps,Tokyo Hibachi Catering,5
2,Kevin was amazing and so fun. I would definite...,1,google_maps,Tokyo Hibachi Catering,5


### 🔍 อธิบาย: ผลลัพธ์ Google Maps Reviews

- ได้ **677 reviews** จริงจาก **10 ร้านอาหาร** — ไม่มี mock data
- Positive (4-5 star): 622 reviews  /  Negative (1-2 star): 55 reviews
- Distribution เอนเอียง Positive เพราะ Google Maps มักมี 5-star reviews เยอะกว่า
  (ผู้ใช้มีแนวโน้มเขียน review เมื่อพอใจมากหรือไม่พอใจมาก)
- ความไม่สมดุลนี้จะถูก balance โดย HuggingFace baseline ที่มี Neg/Pos 50:50


In [7]:
# บันทึก Google Maps reviews ก่อน combine
df_scraped.to_csv('google_maps_reviews_scraped.csv', index=False)
print(f'💾 บันทึกแล้วที่ google_maps_reviews_scraped.csv ({len(df_scraped)} rows)')
df_scraped[['text', 'stars', 'label', 'restaurant']].sample(3, random_state=42)


💾 บันทึกแล้วที่ google_maps_reviews_scraped.csv (677 rows)


,text,stars,label,restaurant
646,While in NY for our anniversary weekend decide...,4,1,The Smith
336,"Service was amazing, food was very good. Spent...",5,1,Boucherie Union Square
63,You wanna have a blast? Get Kevin in your bac...,5,1,Tokyo Hibachi Catering


### 🔍 อธิบาย: รวม Dataset และบันทึกไฟล์ Raw

รวม 2 แหล่งข้อมูลเข้าด้วยกัน:
1. `df_baseline` — ข้อมูลจาก HuggingFace Yelp Polarity (1,000 rows)
2. `df_scraped`  — ข้อมูลจาก Google Maps Reviews (real scraped data)

ขั้นตอน:
- เลือกเฉพาะ columns ที่ตรงกัน (`text`, `label`, `source`)
- ใช้ `pd.concat()` ต่อ DataFrame เข้าด้วยกัน
- ตัด duplicate ด้วย `drop_duplicates(subset=['text'])`
- บันทึกเป็น `raw_combined_reviews.csv`


In [8]:
# ============================================================
# รวม Dataset และ save
# ============================================================

# เลือก columns ให้ตรงกัน
df_baseline_trim = df_baseline[['text', 'label', 'source']]
df_scraped_trim  = df_scraped[['text', 'label', 'source']]

# รวม
df_combined = pd.concat([df_baseline_trim, df_scraped_trim], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['text']).reset_index(drop=True)

# บันทึก raw data
df_combined.to_csv('raw_combined_reviews.csv', index=False)
print(f'💾 Saved raw_combined_reviews.csv — {len(df_combined)} rows')
print(f'\nLabel distribution:')
print(df_combined['label'].value_counts())
print(f'\nSource breakdown:')
print(df_combined['source'].value_counts())


💾 Saved raw_combined_reviews.csv — 1675 rows

Label distribution:
label
1    1110
0     565
Name: count, dtype: int64

Source breakdown:
source
huggingface_baseline    1000
google_maps              675
Name: count, dtype: int64


---
## 🧹 1.2 — Preprocessing Pipeline

Pipeline ประกอบด้วย 6 ขั้นตอน:
1. ลบ HTML tags
2. ลบ URLs
3. ลบ Emojis และ special characters
4. Lowercase + strip
5. Tokenize
6. **ลบ Stopwords** (เช่น `the`, `is`, `a`, `and`, `was` ...)


### 🔍 อธิบาย: ฟังก์ชันทำความสะอาดข้อความ (clean_text)

สร้าง pipeline ทำความสะอาดข้อความใน 5 ขั้นตอน:

| ขั้นตอน | วิธีการ | ตัวอย่าง |
|---|---|---|
| 1. ลบ HTML tags | `re.sub(r'<.*?>', ...)` | `<b>Good</b>` → `Good` |
| 2. ลบ URLs | `re.sub(r'http\S+\|www\.\S+', ...)` | `https://yelp.com` → `` |
| 3. ลบ Emojis | `.encode('ascii', 'ignore')` | `🍕🔥` → `` |
| 4. ลบอักขระพิเศษ | `re.sub(r'[^a-zA-Z0-9...]`, ...)` | `&%$#` → `` |
| 5. Lowercase + trim | `.lower().strip()` | `AMAZING Food` → `amazing food` |

จากนั้นทดสอบกับ 3 ตัวอย่างเพื่อยืนยันว่า pipeline ทำงานถูกต้อง


In [9]:
# ============================================================
# CLEANING PIPELINE
# ============================================================

from nltk.corpus import stopwords

# โหลด English stopwords จาก NLTK
STOPWORDS = set(stopwords.words('english'))

# ดู stopwords ที่ใช้
print(f'Total stopwords: {len(STOPWORDS)}')
print(f'ตัวอย่าง: {sorted(list(STOPWORDS))[:20]}')
print()


def clean_text(text: str) -> str:
    """
    ทำความสะอาด text ก่อน tokenize
    Steps:
      1. Remove HTML tags (ใช้ regex แทน BeautifulSoup)
      2. Remove URLs (http, https, www)
      3. Remove Emojis / Non-ASCII symbols
      4. Remove special characters
      5. Lowercase + strip whitespace
    """
    # 1. ลบ HTML tags ด้วย regex
    text = re.sub(r'<[^>]+>', ' ', text)

    # 2. ลบ URLs (http, https, www)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. ลบ Emojis และ non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 4. ลบอักขระพิเศษ เหลือแค่ตัวอักษร ตัวเลข และ space
    text = re.sub(r"[^a-zA-Z0-9\s'.,!?]", '', text)

    # 5. ลบ whitespace ซ้ำ + lowercase
    text = re.sub(r'\s+', ' ', text).strip().lower()

    return text


def remove_stopwords(text: str) -> str:
    """
    ลบ stopwords ออกจาก text ที่ผ่าน clean_text แล้ว

    ตัวอย่าง stopwords ที่ลบ:
      - Articles:      a, an, the
      - Prepositions:  in, on, at, for, to, from, with, of
      - Conjunctions:  and, or, but, so, yet
      - Auxiliaries:   is, are, was, were, be, been, have, has, had, do, did
      - Pronouns:      i, you, he, she, it, we, they, me, him, her, us, them
      - Common words:  this, that, these, those, there, here, just, also

    คำเหล่านี้ไม่มีผลต่อ sentiment จึงตัดออกเพื่อลด noise ให้ TF-IDF
    """
    tokens = text.split()
    filtered = [word for word in tokens if word not in STOPWORDS]
    return ' '.join(filtered)


# ทดสอบ pipeline
test_cases = [
    '<b>Amazing food!</b> Visit https://yelp.com 🍕🔥',
    'Worst experience EVER!! Staff was <i>rude</i> & slow...',
    'The food was not good at all and the service was terrible.',
]

print('=== Cleaning + Stopword Removal Test ===')
for t in test_cases:
    cleaned  = clean_text(t)
    no_stops = remove_stopwords(cleaned)
    print(f'  IN:       {t}')
    print(f'  CLEANED:  {cleaned}')
    print(f'  NO STOP:  {no_stops}')
    print()


Total stopwords: 198
ตัวอย่าง: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']

=== Cleaning + Stopword Removal Test ===
  IN:       <b>Amazing food!</b> Visit https://yelp.com 🍕🔥
  CLEANED:  amazing food! visit
  NO STOP:  amazing food! visit

  IN:       Worst experience EVER!! Staff was <i>rude</i> & slow...
  CLEANED:  worst experience ever!! staff was rude slow...
  NO STOP:  worst experience ever!! staff rude slow...

  IN:       The food was not good at all and the service was terrible.
  CLEANED:  the food was not good at all and the service was terrible.
  NO STOP:  food good service terrible.



### 🔍 อธิบาย: Apply Cleaning + Stopword Removal ให้ทั้ง Dataset

นำ pipeline ทั้งหมดไป apply กับทุก row ใน `df_combined`:
- **`text_clean`** — ผ่าน `clean_text()` เท่านั้น (HTML, URL, Emoji, lowercase)
- **`text_no_stopwords`** — ผ่าน `remove_stopwords()` เพิ่มเติม (ตัด stopwords ออก)
- กรองทิ้ง row ที่หลัง clean แล้วสั้นกว่า 10 ตัวอักษร
- column `text` เดิมยังเก็บไว้เพื่อ reference เปรียบเทียบ Before/After

**Phase ถัดไปที่ใช้แต่ละ column:**
- `text_clean` → spaCy tokenizer (Phase 3 BiLSTM ใช้ word sequences)
- `text_no_stopwords` → TF-IDF (Phase 3 NB, LR, RF) และ NMF (Phase 2.1)


In [10]:
# Apply cleaning pipeline ให้ทั้ง dataset
print('Applying clean_text()...')
df_combined['text_clean'] = df_combined['text'].apply(clean_text)

# Apply stopword removal
print('Applying remove_stopwords()...')
df_combined['text_no_stopwords'] = df_combined['text_clean'].apply(remove_stopwords)

# ลบ rows ที่หลังทำความสะอาดแล้วสั้นเกิน 10 ตัวอักษร
df_combined = df_combined[
    df_combined['text_clean'].str.len() >= 10
].reset_index(drop=True)

print(f'\n✅ Preprocessing เสร็จ: {len(df_combined)} rows')
print(f'Columns: {list(df_combined.columns)}')

# แสดงตัวอย่าง Before / After
print('\n=== ตัวอย่าง Before / After ===')
sample = df_combined.sample(3, random_state=42)
for _, row in sample.iterrows():
    print(f'  ORIGINAL:    {row["text"][:80]}')
    print(f'  CLEANED:     {row["text_clean"][:80]}')
    print(f'  NO STOPWORD: {row["text_no_stopwords"][:80]}')
    print()

# สถิติ token count
df_combined['token_count_clean']    = df_combined['text_clean'].str.split().str.len()
df_combined['token_count_nostop']   = df_combined['text_no_stopwords'].str.split().str.len()
reduction = (
    1 - df_combined['token_count_nostop'].mean() /
    df_combined['token_count_clean'].mean()
) * 100
print(f'Avg tokens (before stopword removal): {df_combined["token_count_clean"].mean():.1f}')
print(f'Avg tokens (after  stopword removal): {df_combined["token_count_nostop"].mean():.1f}')
print(f'Token reduction: {reduction:.1f}%')
df_combined[['text', 'text_clean', 'text_no_stopwords', 'label']].head(3)


Applying clean_text()...
Applying remove_stopwords()...

✅ Preprocessing เสร็จ: 1653 rows
Columns: ['text', 'label', 'source', 'text_clean', 'text_no_stopwords']

=== ตัวอย่าง Before / After ===
  ORIGINAL:    I will never buy meat ,milk and eggs in this store. I bought shrimps one time, w
  CLEANED:     i will never buy meat ,milk and eggs in this store. i bought shrimps one time, w
  NO STOPWORD: never buy meat ,milk eggs store. bought shrimps one time, got home. opened bags,

  ORIGINAL:    Wir waren heute dort, anl\u00e4sslich unseres Hochzeitstages.\nIch war gespannt,
  CLEANED:     wir waren heute dort, anlu00e4sslich unseres hochzeitstages.nich war gespannt, w
  NO STOPWORD: wir waren heute dort, anlu00e4sslich unseres hochzeitstages.nich war gespannt, w

  ORIGINAL:    Our dinner at Different Pointe of View was perfect. We did the chef's tasting me
  CLEANED:     our dinner at different pointe of view was perfect. we did the chef's tasting me
  NO STOPWORD: dinner different poi

,text,text_clean,text_no_stopwords,label
0,All the steaks are pre-made .. Once you order ...,all the steaks are premade .. once you order y...,steaks premade .. order steak reheat steak dep...,0
1,I was craving Dairy Queen for quite awhile. W...,i was craving dairy queen for quite awhile. wh...,craving dairy queen quite awhile. finally went...,0
2,Un endroit qui ressemble plus \u00e0 un bar sp...,un endroit qui ressemble plus u00e0 un bar spo...,un endroit qui ressemble plus u00e0 un bar spo...,0


---
## 🔤 Tokenization Comparison: NLTK vs spaCy

เปรียบเทียบผลลัพธ์และคุณสมบัติของ tokenizer 2 แบบ

### 🔍 อธิบาย: เปรียบเทียบ Tokenizer NLTK vs spaCy

สร้างฟังก์ชัน tokenizer 2 แบบและเปรียบเทียบผลลัพธ์:

**`tokenize_nltk(text)`**
- ใช้ `word_tokenize()` ตัดคำแบบ rule-based
- เก็บเฉพาะคำที่เป็น alphabet (`isalpha()`) และไม่ใช่ stopword
- ข้อเสีย: ไม่ทำ lemmatization → `running`, `ran`, `runs` ถือเป็นคนละคำ

**`tokenize_spacy(text)`**
- ใช้ `token.lemma_` แปลงคำกลับเป็น base form → `running` → `run`
- กรอง stopword ด้วย `token.is_stop` และ non-alphabet ด้วย `token.is_alpha`
- ข้อดี: vocabulary เล็กลง, โมเดลเรียนรู้ได้ดีขึ้น

เปรียบเทียบบน 3 ประโยคตัวอย่าง แสดงผลเป็นตาราง

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm')
stop_words = set(stopwords.words('english'))

# ============================================================
# NLTK Tokenizer
# ============================================================
def tokenize_nltk(text: str) -> list:
    """
    Tokenize ด้วย NLTK word_tokenize
    - แยก contractions: don't → do, n't
    - ลบ stopwords
    """
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return tokens


# ============================================================
# spaCy Tokenizer
# ============================================================
def tokenize_spacy(text: str) -> list:
    """
    Tokenize ด้วย spaCy
    - ใช้ lemmatization (แปลง running → run)
    - ลบ stopwords และ punctuation
    """
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]
    return tokens


# ============================================================
# เปรียบเทียบผลลัพธ์
# ============================================================
sample_texts = [
    "I don't recommend this place, the food was terrible and overpriced!",
    "Amazing experience! The staff were incredibly friendly and helpful.",
    "The burgers are running hot and fresh every morning.",
]

print('=' * 70)
print('TOKENIZATION COMPARISON: NLTK vs spaCy')
print('=' * 70)

results = []
for text in sample_texts:
    nltk_tokens = tokenize_nltk(text)
    spacy_tokens = tokenize_spacy(text)
    
    results.append({
        'text': text[:60] + '...',
        'NLTK tokens': ' | '.join(nltk_tokens),
        'spaCy tokens (lemmatized)': ' | '.join(spacy_tokens),
        'NLTK count': len(nltk_tokens),
        'spaCy count': len(spacy_tokens),
    })

df_comparison = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 60)
df_comparison

TOKENIZATION COMPARISON: NLTK vs spaCy


,text,NLTK tokens,spaCy tokens (lemmatized),NLTK count,spaCy count
0,"I don't recommend this place, the food was terrible and ...",I | recommend | place | food | terrible | overpriced,recommend | place | food | terrible | overpriced,6,5
1,Amazing experience! The staff were incredibly friendly a...,Amazing | experience | The | staff | incredibly | friend...,amazing | experience | staff | incredibly | friendly | h...,7,6
2,The burgers are running hot and fresh every morning....,The | burgers | running | hot | fresh | every | morning,burger | run | hot | fresh | morning,7,5


### 🔍 อธิบาย: ตารางสรุปความแตกต่าง NLTK vs spaCy

สรุปความแตกต่างในเชิง feature ระหว่าง 2 tokenizer:
- **Speed** — spaCy เร็วกว่าเพราะ backend เขียนด้วย C
- **Lemmatization** — spaCy มี built-in, NLTK ต้อง import `WordNetLemmatizer` แยก
- **POS / NER** — spaCy มีครบ ซึ่งจะใช้ใน Phase 2 (NER Task)

**สรุป:** เลือกใช้ **spaCy** เป็น tokenizer หลักสำหรับ Phase ถัดไป

In [12]:
# ============================================================
# ตารางเปรียบเทียบ Feature
# ============================================================
comparison_table = {
    'Feature': [
        'Speed',
        'Contraction Handling',
        'Lemmatization',
        'POS Tagging',
        'NER Support',
        'Best For'
    ],
    'NLTK': [
        'ช้ากว่า (pure Python)',
        "แยก don't → do + n't",
        'ต้อง WordNetLemmatizer แยก',
        'ต้อง import แยก',
        'ไม่มี built-in',
        'Baseline / Prototype'
    ],
    'spaCy': [
        'เร็วกว่า (C-optimized)',
        "แยก don't → do + n't + linguistic info",
        'Built-in (lemma_ attribute)',
        'Built-in (token.pos_)',
        'Built-in (Phase 2 NER)',
        'Production / Phase 2-3'
    ]
}
pd.DataFrame(comparison_table)

,Feature,NLTK,spaCy
0,Speed,ช้ากว่า (pure Python),เร็วกว่า (C-optimized)
1,Contraction Handling,แยก don't → do + n't,แยก don't → do + n't + linguistic info
2,Lemmatization,ต้อง WordNetLemmatizer แยก,Built-in (lemma_ attribute)
3,POS Tagging,ต้อง import แยก,Built-in (token.pos_)
4,NER Support,ไม่มี built-in,Built-in (Phase 2 NER)
5,Best For,Baseline / Prototype,Production / Phase 2-3


### 🔍 อธิบาย: Speed Benchmark บน Dataset จริง

วัดเวลาจริงของทั้ง 2 tokenizer บน 500 reviews:
- ใช้ `time.time()` จับเวลาก่อน-หลัง
- เปรียบเทียบทั้ง **เวลาที่ใช้** และ **จำนวน token เฉลี่ยต่อ review**
- spaCy มักได้จำนวน token น้อยกว่า NLTK เพราะ lemmatization รวมรูปคำต่างๆ เข้าด้วยกัน

> ผลที่ได้นำไปใส่ในรายงานเปรียบเทียบ tokenizer ได้เลย

In [13]:
import time

# ============================================================
# Speed Benchmark บน 500 reviews จริง
# ============================================================
sample_500 = df_combined['text_clean'].head(500).tolist()

# NLTK
t0 = time.time()
nltk_results = [tokenize_nltk(t) for t in sample_500]
nltk_time = time.time() - t0

# spaCy
t0 = time.time()
spacy_results = [tokenize_spacy(t) for t in sample_500]
spacy_time = time.time() - t0

print(f'⏱️  Speed Benchmark (500 texts):')
print(f'   NLTK:  {nltk_time:.2f} seconds')
print(f'   spaCy: {spacy_time:.2f} seconds')
print(f'   Winner: {"NLTK" if nltk_time < spacy_time else "spaCy"} 🏆')

avg_nltk_len  = sum(len(t) for t in nltk_results) / len(nltk_results)
avg_spacy_len = sum(len(t) for t in spacy_results) / len(spacy_results)
print(f'\n📊 Average token count per review:')
print(f'   NLTK:  {avg_nltk_len:.1f} tokens')
print(f'   spaCy: {avg_spacy_len:.1f} tokens (fewer = lemmatization collapsed forms)')

⏱️  Speed Benchmark (500 texts):
   NLTK:  0.29 seconds
   spaCy: 8.30 seconds
   Winner: NLTK 🏆

📊 Average token count per review:
   NLTK:  61.7 tokens
   spaCy: 53.3 tokens (fewer = lemmatization collapsed forms)


### 🔍 อธิบาย: Apply Tokenization และบันทึกไฟล์สุดท้าย

Apply `tokenize_spacy()` กับทุก review ใน dataset:
- แปลง list of tokens กลับเป็น string ด้วย `' '.join(...)` เพื่อเก็บใน CSV ได้
- บันทึกเป็น `preprocessed_reviews.csv` ซึ่งจะใช้เป็น input ของ Phase 2 และ 3

**Columns ในไฟล์สุดท้าย:**
- `text` — ข้อความดิบต้นฉบับ
- `text_clean` — ข้อความหลังทำความสะอาด
- `tokens` — คำหลังผ่าน tokenization + lemmatization
- `label` — 0 = Negative, 1 = Positive
- `source` — แหล่งที่มาของข้อมูล

In [14]:
# ============================================================
# Apply tokenization ที่เลือกใช้จริง (spaCy) และ save
# ============================================================

# ใช้ spaCy เป็น final tokenizer (lemmatized, stopword removed)
print('Applying spaCy tokenization to full dataset...')
df_combined['tokens'] = df_combined['text_clean'].apply(
    lambda x: ' '.join(tokenize_spacy(x))
)

# Save final preprocessed dataset
df_combined.to_csv('preprocessed_reviews.csv', index=False)
print(f'\n💾 Saved preprocessed_reviews.csv — {len(df_combined)} rows')
print('\nColumns:', df_combined.columns.tolist())
df_combined[['text_clean', 'tokens', 'label']].head(3)

Applying spaCy tokenization to full dataset...

💾 Saved preprocessed_reviews.csv — 1653 rows

Columns: ['text', 'label', 'source', 'text_clean', 'text_no_stopwords', 'token_count_clean', 'token_count_nostop', 'tokens']


,text_clean,tokens,label
0,all the steaks are premade .. once you order your steak ...,steak premade order steak reheat steak depend request ra...,0
1,i was craving dairy queen for quite awhile. when i final...,crave dairy queen awhile finally go wonder want badly or...,0
2,un endroit qui ressemble plus u00e0 un bar sportif qu'u0...,un endroit qui ressemble plus un bar sportif un est bon ...,0


---
## 📊 Summary: Phase 1

| หัวข้อ | รายละเอียด |
|---|---|
| Standard Dataset | Yelp Polarity (HuggingFace) — 1,000 reviews |
| Web Scraped | Apify — 500-600 reviews (10 ร้าน) |
| Total (หลัง clean) | ≥ 1500 reviews |
| Cleaning | HTML, URLs, Emojis, Lowercase |
| Tokenizer เลือกใช้ | spaCy (lemmatized + stopword removed) |
| Output | `preprocessed_reviews.csv` |

### 🔍 อธิบาย: สรุปผล Phase 1

แสดงสถิติสรุปของ dataset ที่เตรียมเสร็จแล้ว:
- จำนวน reviews ทั้งหมด
- จำนวน Positive vs Negative (ควรใกล้เคียงกันเพื่อป้องกัน class imbalance)
- ความยาวเฉลี่ยของข้อความและ token
- รายชื่อไฟล์ที่บันทึกไว้

In [15]:
# สรุปสุดท้าย
print('=' * 50)
print('PHASE 1 COMPLETE ✅')
print('=' * 50)
print(f'Total reviews       : {len(df_combined)}')
print(f'Positive (label=1)  : {(df_combined["label"]==1).sum()}')
print(f'Negative (label=0)  : {(df_combined["label"]==0).sum()}')
print(f'Avg text length     : {df_combined["text_clean"].str.len().mean():.0f} chars')
print(f'Avg token count     : {df_combined["tokens"].str.split().apply(len).mean():.0f} tokens')
print('\nFiles saved:')
print('  📄 raw_combined_reviews.csv')
print('  📄 preprocessed_reviews.csv')

PHASE 1 COMPLETE ✅
Total reviews       : 1653
Positive (label=1)  : 1090
Negative (label=0)  : 563
Avg text length     : 502 chars
Avg token count     : 40 tokens

Files saved:
  📄 raw_combined_reviews.csv
  📄 preprocessed_reviews.csv


---
## 📝 AI Audit Log — Phase 1

| Task | Prompt ที่ใช้ | ผลลัพธ์จาก AI | การตรวจสอบและแก้ไข (Human Verification) |
|---|---|---|---|
| Regex Cleaning | "Write regex to remove URLs from review text" | `r'http\S+'` | **Fail:** ไม่ครอบคลุม `www.` URLs<br>**Fix:** เพิ่ม `r'http\S+\|www\.\S+'` |
| Star Rating Parser | "Parse star rating from Yelp aria-label attribute" | `re.search(r'(\d)', tag['aria-label'])` | **Pass w/ Edit:** ทำงานได้แต่ต้อง fallback 2 วิธีเพราะ Yelp เปลี่ยน class<br>**Fix:** เพิ่ม fallback `role='img'` |
| spaCy Tokenizer | "Tokenize and lemmatize text removing stopwords with spaCy" | โค้ด list comprehension พื้นฐาน | **Pass:** ทำงานได้ แต่เพิ่ม `token.is_alpha` เพื่อกรองตัวเลขออก |
| Pagination Logic | "How to paginate Yelp reviews with requests" | `?start=10` per page | **Pass:** ตรงกับ Yelp URL จริง ยืนยันด้วย manual inspection |
| DataFrame Safety | "Convert scraped list to DataFrame" | `pd.DataFrame(all_scraped)` แล้วใช้ `df['label']` ทันที | **Fail:** `KeyError: 'label'` เมื่อ Yelp block และ list ว่าง<br>**Fix:** เพิ่ม `if df_scraped.empty or 'label' not in df_scraped.columns` guard + mock data fallback |
| Business IDs | "List popular Yelp restaurant business IDs in New York" | ID คาดเดาจาก pattern เช่น `joes-pizza-new-york-41` | **Fail:** ค้นหาใน Yelp จริงไม่เจอ เพราะ ID คิดขึ้นมาเอง<br>**Fix:** verify จาก Yelp search จริง แล้วเปลี่ยนเป็น ID ที่ถูกต้อง เช่น `julianas-brooklyn-3` |